# dev 40장 — 같은 부품 영역 비교

**전체 실행을 누르지 말고, 박스 표시 단계에서 멈춰 작업하세요.**

- 같은 dev 40장 / Q1·Q2 80문항을 사용합니다. 기본 모델 InternVL, 전체 이미지와 부품 crop 두 조건(최대160호출).
- 정상 참조와 대상에서 **같은 종류의 검사 부품 전체**에 박스를 하나씩 그립니다. 결함만 좁게 표시하지 말고 주변 구조를 포함하세요. 정상 사진에도 같은 기준을 적용하세요.
- 박스 화면에 정답/모델 답/원본 파일명을 표시하지 않습니다. 대응 부품을 지정할 수 없으면 사유와 함께 저장하고 두 조건에서 모두 제외합니다. 제외 수와 비교 대상 수를 별도로 보고합니다.
- 매 사진 쌍 저장 후 Drive에 기록합니다. 끊겨도 첫 미완료부터 이어갑니다.
- 이미지 두 장 순서(정상 참조→대상), 질문·보기·기준선 프롬프트·추론 설정은 같습니다. 이전 답/재검토 문구/정답 위치 문구는 모델에 주지 않습니다.
- 기존 결과/gold/설정은 보존. 출력은 `MyDrive/lvlm_insplad_results/gold_v2_part_crop_diagnostic_v1/`. 자동 게시·설정 선택 없음.
- 이 결과는 **수작업 부품 영역을 이용한 가능성 진단**입니다. 자동 영역 탐지 성능이나 gold 성능으로 보고하면 안 됩니다. 두 조건의 CI는 작은 dev 탐색용입니다.

순서: 설정→준비→이미지 준비→박스 표시→완료 확인→GPU 준비→추론→결과→ZIP 다운로드.
CPU에서 표시해도 됩니다. 추론 전에 L4로 바꾸면 런타임이 초기화되므로 준비 셀부터 다시 실행하세요(저장된 박스는 유지). 편하게 한 번에 진행하려면 처음부터 L4로 설정하세요.


In [ ]:
REPO = "arnold6444/lvlm_insplad"
REPO_REF = "main"
DEV_FOLDER = "lvlm_insplad_qa/gold_v2_dev"
DEV_BASELINES_FOLDER = "lvlm_insplad_results/gold_v2_dev_baselines"
RUN_FOLDER = "lvlm_insplad_results/gold_v2_part_crop_diagnostic_v1"
MODELS = ["internvl3_5_8b"]
N_IMAGES = 40
SEED = 42
CHECKPOINT_EVERY = 20
UNSUP_ZIP_NAME = "unsupervised_anomaly_detection.zip"
UNSUP_ZIP_SIZE = 1_170_967_719
UNSUP_ZIP_PATH = ""
SUP_ZIP_NAME = "supervised_fault_classification.zip"
SUP_ZIP_SIZE = 874_753_167
SUP_ZIP_PATH = ""


In [ ]:
# Drive 연결 + repo clone + 패키지
import base64, json, os, shutil, subprocess, sys
from pathlib import Path
from google.colab import drive, userdata

drive.mount("/content/drive", force_remount=False)
DRIVE_ROOT = Path("/content/drive/MyDrive")
DEV_DIR = DRIVE_ROOT / DEV_FOLDER
RUN_ROOT = DRIVE_ROOT / RUN_FOLDER
RUN_ROOT.mkdir(parents=True, exist_ok=True)
DEV_REFS_DIR = DEV_DIR / "refs"
STAGE1_DIR = DRIVE_ROOT / DEV_BASELINES_FOLDER / "predictions"

REPO_DIR = Path("/content/lvlm_part_crop_code")

def run(cmd, *, env=None, check=True):
    """Run a command and stream its output into this cell (Colab hides plain subprocess output)."""
    cmd = [str(x) for x in cmd]
    print("+", " ".join(cmd[:12]) + (" ..." if len(cmd) > 12 else ""), flush=True)
    p = subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    tail = []
    for line in p.stdout:
        tail = (tail + [line])[-60:]
        print(line, end="", flush=True)
    p.wait()
    if check and p.returncode:
        raise RuntimeError(f"command failed with exit code {p.returncode} (마지막 출력 위 참고)")
    return p.returncode

github_token = userdata.get("GITHUB_TOKEN")
if not github_token:
    raise RuntimeError("Colab Secret GITHUB_TOKEN이 필요합니다.")
os.environ["GITHUB_TOKEN"] = github_token
auth = base64.b64encode(f"x-access-token:{github_token}".encode()).decode()
git_env = os.environ.copy()
git_env.update({"GIT_CONFIG_COUNT":"1", "GIT_CONFIG_KEY_0":"http.extraHeader",
                "GIT_CONFIG_VALUE_0":f"AUTHORIZATION: basic {auth}", "GIT_TERMINAL_PROMPT":"0"})
if not REPO_DIR.exists():
    run(["git","clone","--depth","1","--branch",REPO_REF,f"https://github.com/{REPO}.git",REPO_DIR], env=git_env)
del auth, git_env
run([sys.executable,"-m","pip","install","-q","pillow>=10","pandas>=2.2"])
sys.path.insert(0, str(REPO_DIR / "src"))
sys.path.insert(0, str(REPO_DIR / "scripts"))
import pandas as pd
print("repo:", subprocess.check_output(["git","rev-parse","HEAD"], cwd=REPO_DIR, text=True).strip())


In [ ]:
def find_zip(filename, expected_size, override=""):
    candidates = [Path(override)] if override else []
    candidates += [DRIVE_ROOT / filename, DRIVE_ROOT / "lvlm_insplad_data" / filename]
    for p in candidates:
        if p.is_file() and p.stat().st_size == expected_size:
            return p
    print(f"{filename}: Drive 전체 검색 중 (몇 분 걸릴 수 있음)", flush=True)
    for p in DRIVE_ROOT.rglob(filename):
        if p.is_file() and p.stat().st_size == expected_size:
            return p
    raise FileNotFoundError(f"{filename} ({expected_size} bytes) not found")

SUP_ZIP = find_zip(SUP_ZIP_NAME, SUP_ZIP_SIZE, SUP_ZIP_PATH)
UNSUP_ZIP = find_zip(UNSUP_ZIP_NAME, UNSUP_ZIP_SIZE, UNSUP_ZIP_PATH)
print("SUP:", SUP_ZIP); print("UNSUP:", UNSUP_ZIP)

summary = json.loads((DEV_DIR / "build_summary.json").read_text(encoding="utf-8"))
if summary.get("quality_status") != "FROZEN":
    raise RuntimeError(f"dev 세트가 FROZEN이 아닙니다: {summary.get('quality_status')}")
DEV_QA = DEV_DIR / summary["file"]
REFS_CSV = DEV_REFS_DIR / "retrieved_refs.csv"
need = [DEV_QA, REFS_CSV] + [STAGE1_DIR / m / "normal_ref_retrieved_diffdate.jsonl" for m in MODELS]
missing = [str(p) for p in need if not p.is_file()]
if missing:
    raise FileNotFoundError("기존 dev 기준선 입력 없음: " + ", ".join(missing))
import dev_part_crop_annotation as part
import run_dev_part_crop_diagnostic as comparison
rows, pairs = part.prepare(DEV_QA, REFS_CSV, SUP_ZIP, UNSUP_ZIP, RUN_ROOT, N_IMAGES, SEED)
session = part.PartSession(pairs, RUN_ROOT / "annotations.json")
print("준비 완료:", session.progress())


## 박스 표시
양쪽 부품을 지정하고 저장하세요. 완료 후 다음 셀을 실행합니다.


In [ ]:
# 이 셀 출력 화면에서 양쪽 부품 박스를 드래그하고 '저장 후 다음'을 누르세요.
# 모든 사진을 저장할 때까지 다음 셀로 넘어가지 마세요.
part.register_colab(session)


In [ ]:
# 표시 완료 확인. 미완료가 있으면 이 셀에서 중단됩니다.
session = part.PartSession(pairs, RUN_ROOT / "annotations.json")
eligible, excluded = session.completed()
print(f"비교 대상 {len(eligible)}/{len(pairs)}장, 지정 불가 {len(excluded)}장")
print(f"추론 호출 {len(eligible)*2*2*len(MODELS)}회")
from IPython.display import display
from PIL import Image
for pair in eligible[:3]:
    paths = comparison.crop_pair(pair, session.data[pair["core_id"]], RUN_ROOT / "_crops")
    print(pair["asset_category"], "정상 참조 / 대상 crop 미리보기")
    thumbnails = []
    for path in paths:
        with Image.open(path) as im:
            thumbnail = im.convert("RGB")
            thumbnail.thumbnail((450, 300))
            thumbnails.append(thumbnail)
    joined = Image.new("RGB", (sum(im.width for im in thumbnails)+10, max(im.height for im in thumbnails)), "white")
    joined.paste(thumbnails[0], (0, 0)); joined.paste(thumbnails[1], (thumbnails[0].width+10, 0))
    display(joined)


In [ ]:
# 박스 표시 후 GPU 준비. 미완료 상태에서는 설치/모델 호출을 시작하지 않습니다.
part.PartSession(pairs, RUN_ROOT / "annotations.json").completed()
import torch
if not torch.cuda.is_available() or torch.cuda.get_device_properties(0).total_memory < 20*1024**3:
    raise RuntimeError("L4/A100 런타임이 필요합니다. 변경 후 준비 셀부터 다시 실행하세요. 박스는 Drive에 남습니다.")
run([sys.executable,"-m","pip","install","-q","transformers==5.14.0","accelerate==1.14.0",
     "sentencepiece==0.2.2","pandas>=2.2","numpy>=1.26","pillow>=10","tqdm>=4.66","timm"])
print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
# 추론. 끊기면 이 셀만 다시 실행: 성공한 호출은 재사용, 오류 호출만 재시도.
run([sys.executable, REPO_DIR / "scripts/run_dev_part_crop_diagnostic.py",
     "--workspace", RUN_ROOT, "--stage1-dir", STAGE1_DIR, "--models", *MODELS,
     "--checkpoint-every", str(CHECKPOINT_EVERY)])


In [ ]:
RESULT_DIR = RUN_ROOT / "inference"
status = json.loads((RESULT_DIR / "run_status.json").read_text(encoding="utf-8"))
coverage = json.loads((RESULT_DIR / "coverage.json").read_text(encoding="utf-8"))
print("row_errors 합계:", sum(s["row_errors"] for s in status))
print("무효 답변 합계:", sum(s["invalid_answers"] for s in status))
print("선택/비교/제외:", coverage["selected_images"], coverage["included_images"], coverage["excluded_images"])
display(pd.DataFrame(coverage["included_by_asset_gt"]))
display(pd.DataFrame(status))
lb = pd.read_csv(RESULT_DIR / "leaderboard.csv")
display(lb[["model_key","condition","q1_anomaly_f1","q2_defect_macro_f1","q1_accuracy","q2_accuracy",
            "joint_accuracy","false_defect_rate","old_baseline_agreement","latency_s"]].round(3))
display(pd.read_csv(RESULT_DIR / "significance.csv").round(3))
answers = pd.read_csv(RESULT_DIR / "answers.csv")
q1 = answers[answers.question_type == "anomaly_detection"]
display(pd.crosstab([q1.model_key, q1.condition], q1.predicted_text).reindex(columns=["Normal.","Anomalous."],fill_value=0))
print("whole_image의 old_baseline_agreement=1이면 이전 기준선 답이 재현된 것입니다.")
print("수작업 crop으로 비교한 포함 이미지의 결과입니다. 자동화·gold 성능으로 해석하지 마세요.")


In [ ]:
# 결과 ZIP 다운로드 (Drive 결과는 그대로 유지)
from zipfile import ZipFile, ZIP_DEFLATED
from google.colab import files
archive = Path("/content/dev_part_crop_diagnostic_results.zip")
with ZipFile(archive, "w", ZIP_DEFLATED) as z:
    for p in RUN_ROOT.rglob("*"):
        if p.is_file() and not ({"_images", "_crops"} & set(p.relative_to(RUN_ROOT).parts)):
            z.write(p, p.relative_to(RUN_ROOT))
files.download(str(archive))
